# 버스 노선별 경유 정류장정보 조회 및 수요 데이터 정류장명 매핑

03에서 수집한 2024-10-17·2025-10-16 경기 승차·서울 하차 로우 데이터의 실제 노선ID와 승차·하차 정류장ID를 STCIS 버스 노선별 경유정류장정보 API로 매핑한다.

### 셀 1. 노선별 경유 정류장 API 기본 조회\n

### ?? ? 1. STCIS ??? API ?? ?? ? ???? ?? ??


In [ ]:
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "gtx_a_seoul_bus_outputs" / "transport_card"
STOP_DIR = DATA_DIR / "route_stops"
STOP_DIR.mkdir(parents=True, exist_ok=True)

API_URL = "https://stcis.go.kr/openapi/busroutesttn.json"
API_KEY = getpass("STCIS 인증키를 입력하세요: ").strip()
REGION_CODES = ["41", "11"]  # 경기도 + 서울
DATES = ["20241017", "20251016"]

if not API_KEY:
    raise ValueError("STCIS 인증키가 입력되지 않았습니다.")

def parse_response(payload):
    status = payload.get("status", "")
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = [result]
    if isinstance(result, list):
        return result, status
    return [], status

def get_route_stops(route_id, sd_cd):
    params = {"apikey": API_KEY, "sdCd": sd_cd, "routeId": route_id}
    response = requests.get(API_URL, params=params, timeout=30)
    response.raise_for_status()
    return parse_response(response.json())

def load_raw(date):
    path = DATA_DIR / f"gtx_a_transport_card_{date}_raw.csv"
    if not path.exists():
        raise FileNotFoundError(f"03 실행 결과가 없습니다: {path}")
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig").fillna("")
    for col in ["query_route_id", "query_route_no", "ride_sttn_id", "goff_sttn_id"]:
        if col not in df.columns:
            raise ValueError(f"{path.name}에 {col} 컬럼이 없습니다.")
        df[col] = df[col].astype(str).str.strip()
    return df

def collect_stops(date, raw):
    routes = raw[["query_route_id", "query_route_no"]].drop_duplicates()
    records, logs = [], []

    for i, route in routes.reset_index(drop=True).iterrows():
        route_id = route["query_route_id"]
        route_no = route["query_route_no"]
        try:
            region_items = []
            statuses = []
            for sd_cd in REGION_CODES:
                items, status = get_route_stops(route_id, sd_cd)
                statuses.append(f"{sd_cd}:{status or 'OK'}")
                region_items.extend(items)
            # 서울·경기 양쪽 응답을 합치고 동일 정류장은 제거
            seen_stops = set()
            for item in region_items:
                item = dict(item)
                stop_key = (str(item.get("sttnId", "")).strip(), str(item.get("sttnSeq", "")).strip())
                if stop_key in seen_stops:
                    continue
                seen_stops.add(stop_key)
                item.update({
                    "query_date": date,
                    "query_route_id": route_id,
                    "query_route_no": route_no,
                })
                records.append(item)
            items = region_items
            logs.append({
                "query_date": date, "route_id": route_id, "route_no": route_no,
                "status": ";".join(statuses), "count": len(items), "error": ""
            })
            print(f"[{i + 1}/{len(routes)}] {route_no} ({route_id}): {len(items)}개 정류장")
        except Exception as exc:
            logs.append({
                "query_date": date, "route_id": route_id, "route_no": route_no,
                "status": "ERROR", "count": 0, "error": repr(exc)
            })
            print(f"[{i + 1}/{len(routes)}] {route_no} ({route_id}): 실패 - {exc}")
        time.sleep(0.2)

    stops = pd.DataFrame(records)
    if not stops.empty:
        stops["_sttn_seq_num"] = pd.to_numeric(stops.get("sttnSeq"), errors="coerce")
        stops = (stops.drop_duplicates(["query_date", "query_route_id", "sttnId", "sttnSeq"])
                      .sort_values(["query_date", "query_route_id", "_sttn_seq_num"])
                      .drop(columns=["_sttn_seq_num"]))
    logs_df = pd.DataFrame(logs)
    stops_path = STOP_DIR / f"gtx_a_route_stops_{date}.csv"
    stops.to_csv(stops_path, index=False, encoding="utf-8-sig")
    print(f"정류장 원본 저장: {stops_path} ({len(stops)}건)")
    return stops, logs_df

def enrich_with_stops(raw, stops, date):
    if stops.empty:
        print(f"{date}: 정류장 응답이 없어 매핑을 건너뜁니다.")
        return raw

    stops = stops.copy()
    stops = stops.rename(columns={
        "routeId": "api_route_id",
        "routeNo": "api_route_no",
        "sttnSeq": "sttn_seq",
        "sttnId": "sttn_id",
        "sttnNm": "sttn_nm",
        "sdCd": "sttn_sd_cd",
        "sggCd": "sttn_sgg_cd",
        "emdCd": "sttn_emd_cd",
        "sdNm": "sttn_sd_nm",
        "sggNm": "sttn_sgg_nm",
        "emdNm": "sttn_emd_nm",
    })
    required = ["query_route_id", "sttn_id", "sttn_nm"]
    missing = [c for c in required if c not in stops.columns]
    if missing:
        raise ValueError(f"{date} 정류장 응답에 필요한 컬럼이 없습니다: {missing}")

    stops["query_route_id"] = stops["query_route_id"].astype(str).str.strip()
    stops["sttn_id"] = stops["sttn_id"].astype(str).str.strip()
    ref = stops.drop_duplicates(["query_route_id", "sttn_id"]).copy()

    ride_ref = ref.rename(columns={
        c: f"ride_{c}" for c in ref.columns
        if c not in ["query_route_id", "sttn_id"]
    }).rename(columns={
        "query_route_id": "ride_query_route_id",
        "sttn_id": "ride_sttn_id",
    })
    result = raw.merge(
        ride_ref,
        left_on=["query_route_id", "ride_sttn_id"],
        right_on=["ride_query_route_id", "ride_sttn_id"],
        how="left",
    )

    goff_ref = ref.rename(columns={
        c: f"goff_{c}" for c in ref.columns
        if c not in ["query_route_id", "sttn_id"]
    }).rename(columns={
        "query_route_id": "goff_query_route_id",
        "sttn_id": "goff_sttn_id",
    })
    result = result.merge(
        goff_ref,
        left_on=["query_route_id", "goff_sttn_id"],
        right_on=["goff_query_route_id", "goff_sttn_id"],
        how="left",
    )

    result = result.drop(
        columns=[c for c in ["ride_query_route_id", "goff_query_route_id"] if c in result.columns]
    )
    result["승차정류장명"] = result.get("ride_sttn_nm", "")
    result["하차정류장명"] = result.get("goff_sttn_nm", "")
    # ID 바로 옆에 한글 정류장명이 오도록 컬럼 순서 정리
    front = ["query_date", "query_route_id", "query_route_no", "ride_ctpv_cd",
             "승차정류장ID", "승차정류장명", "goff_ctpv_cd", "하차정류장ID", "하차정류장명"]
    result = result.rename(columns={"ride_sttn_id": "승차정류장ID", "goff_sttn_id": "하차정류장ID"})
    front = [c for c in front if c in result.columns]
    result = result[front + [c for c in result.columns if c not in front]]
    return result

results = {}
for date in DATES:
    print(f"\n===== {date} 정류장 조회 =====")
    raw = load_raw(date)
    stops, logs = collect_stops(date, raw)
    enriched = enrich_with_stops(raw, stops, date)

    enriched_path = DATA_DIR / f"gtx_a_transport_card_{date}_raw_with_stop_names.csv"
    enriched.to_csv(enriched_path, index=False, encoding="utf-8-sig")
    results[date] = enriched
    print(f"수요 정류장명 매핑 저장: {enriched_path} ({len(enriched)}건)")
    print(f"승차 정류장명 매칭률: {enriched['승차정류장명'].ne('').mean():.1%}")
    print(f"하차 정류장명 매칭률: {enriched['하차정류장명'].ne('').mean():.1%}")

### 셀 2. 9030·9030-1·M7111 추가 노선의 정류장 조회\n

### ?? ? 2. 2024? ?? 3? ??? ?? ??? ??


In [ ]:
# 2024-10-17 추가 3개 노선만 경유 정류장 조회 (이 셀만 단독 실행 가능)
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

extra_data_dir = Path.cwd() / 'gtx_a_seoul_bus_outputs' / 'transport_card'
# 기존 route_stops 폴더는 건드리지 않고 transport_card 바로 아래에 별도 저장
extra_stop_dir = extra_data_dir
extra_stop_dir.mkdir(parents=True, exist_ok=True)
extra_url = 'https://stcis.go.kr/openapi/busroutesttn.json'
extra_key = getpass('STCIS 인증키를 입력하세요: ').strip()
# 2024년 과거 ID는 경유 정류장 API에서 NOT_FOUND이므로 2025년 ID의 경로를 임시 사용
extra_routes = {'9030': '41084004', '9030-1': '41084005', 'M7111': '41016903'}
extra_rows = []

def extra_parse_stops(payload):
    result = payload.get('result', [])
    if isinstance(result, dict): result = [result]
    return result if isinstance(result, list) else [], payload.get('status', '')

for route_no, route_id in extra_routes.items():
    route_rows = []
    statuses = []
    for sd_cd in ['41', '11']:
        params = {'apikey': extra_key, 'sdCd': sd_cd, 'routeId': route_id}
        try:
            response = requests.get(extra_url, params=params, timeout=60)
            response.raise_for_status()
            items, status = extra_parse_stops(response.json())
            statuses.append(f"{sd_cd}:{status or 'OK'}")
            route_rows.extend(items)
        except Exception as exc:
            statuses.append(f'{sd_cd}:ERROR')
            print(f'{route_no} ({route_id}) / sdCd={sd_cd} 실패: {exc}')
        time.sleep(2)
    seen = set()
    for item in route_rows:
        item = dict(item)
        key = (str(item.get('sttnId', '')).strip(), str(item.get('sttnSeq', '')).strip())
        if key in seen: continue
        seen.add(key)
        item.update({'query_date': '20241017', 'source_route_date': '20251016', 'query_route_id': route_id, 'query_route_no': route_no})
        extra_rows.append(item)
    pd.DataFrame(extra_rows).to_csv(extra_stop_dir / 'gtx_a_route_stops_20241017_extra_3routes.csv', index=False, encoding='utf-8-sig')
    print(f"{route_no} ({route_id}): {len(route_rows)}개 정류장 / {';'.join(statuses)}")

extra_stops = pd.DataFrame(extra_rows)
extra_path = extra_stop_dir / 'gtx_a_route_stops_20241017_extra_3routes.csv'
extra_stops.to_csv(extra_path, index=False, encoding='utf-8-sig')
print(f'추가 3개 노선 정류장 저장: {extra_path} ({len(extra_stops)}건)')


### 셀 3. 추가 노선 교통카드 데이터에 정류장명·순서 매핑\n

### ?? ? 3. ?? 3? ??? ???? ???? ??????? ??


In [ ]:
# 추가 3개 노선의 교통카드 로우에 정류장명·순서 매핑
from pathlib import Path
import pandas as pd

map_dir = Path.cwd() / 'gtx_a_seoul_bus_outputs' / 'transport_card'
raw_path = map_dir / 'gtx_a_transport_card_20241017_raw.csv'
stops_path = map_dir / 'gtx_a_route_stops_20241017_extra_3routes.csv'
out_path = map_dir / 'gtx_a_transport_card_20241017_raw_extra_3routes_with_stop_names.csv'
target_routes = ['9030', '9030-1', 'M7111']

raw = pd.read_csv(raw_path, dtype=str, encoding='utf-8-sig').fillna('')
raw = raw[raw['query_route_no'].isin(target_routes)].copy()
stops = pd.read_csv(stops_path, dtype=str, encoding='utf-8-sig').fillna('')
stops['query_route_no'] = stops['query_route_no'].astype(str).str.strip()
stops['sttnId'] = stops['sttnId'].astype(str).str.strip()
stops['sttnSeq'] = stops['sttnSeq'].astype(str).str.strip()
ref = stops[['query_route_no', 'sttnId', 'sttnNm', 'sttnSeq']].drop_duplicates(['query_route_no', 'sttnId'])
ride_ref = ref.rename(columns={'sttnId': 'ride_sttn_id', 'sttnNm': '승차정류장명', 'sttnSeq': '승차정류장순서'})
goff_ref = ref.rename(columns={'sttnId': 'goff_sttn_id', 'sttnNm': '하차정류장명', 'sttnSeq': '하차정류장순서'})
result = raw.merge(ride_ref, on=['query_route_no', 'ride_sttn_id'], how='left')
result = result.merge(goff_ref, on=['query_route_no', 'goff_sttn_id'], how='left')
front = ['query_date', 'query_route_no', 'query_route_id', 'ride_sttn_id', '승차정류장명', '승차정류장순서', 'goff_sttn_id', '하차정류장명', '하차정류장순서']
result = result[[c for c in front if c in result.columns] + [c for c in result.columns if c not in front]]
result.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'3개 노선 정류장명 매핑 저장: {out_path} ({len(result)}건)')
print(f'승차명 매칭률: {result["승차정류장명"].ne("").mean():.1%}')
print(f'하차명 매칭률: {result["하차정류장명"].ne("").mean():.1%}')
